 add written info on top about timestamps
 need to add rto number, reason, define things in a readme, start making it 

 for baseline, are we looking for products that were returned,  that is there with the row and we don't try to find the correlation between factors that lead to return

 production baseline can have a few snapshots for dark stores everyday and a final eod snapshot
 
 and a single eod snapshot for hubs 



 ---
 what i changed now:
 added rto reason, seasonality trends for 31 day period. 
 **flaws i am aware of:**
 didn't factor in the holiays, the increase factor for weekends remains same for each product, and it is same for each week. 

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [2]:
### Load the data
df_prod = pd.read_csv('01_Test_products.csv')
df_node = pd.read_csv('01_Test_nodes.csv')

In [3]:
records = []
start_date = datetime(2026,3,1) 
for day in range(31):
    curr_date = start_date + timedelta(days=day)
    ### Month-start and weekends sale hikes
    is_weekend = curr_date.weekday() in [5,6]
    is_month_start = curr_date.day <=3

    for _, node in df_node.iterrows():
        for _, prod in df_prod.iterrows():
            ### 1. Baseline Logic: Hubs will have large stock and Darkstores are going to have smaller stock
            if node['type'] == 'Hub':
                stock = np.random.randint(5000,10000)
                base_sales = np.random.randint(100,500)
            else:
                stock = np.random.randint(20,150)
                base_sales = np.random.randint(5,30)

            ### 2. factor-in the sales hike during weekends and month start
            if is_weekend:
                sales = int(base_sales * 1.3)
            elif is_month_start:
                sales = int(base_sales * 1.4)
            else:
                sales = base_sales
            ### 3. RTO logic
            units_rto = np.random.randint(0,sales//5)
            if units_rto > 0:
                rto_reason = np.random.choice(['cancelled', 'customer unavailable', 'incorrect address'])
            else:
                rto_reason = 'None'
            ### append the record
            records.append({
                'timestamp': curr_date,
                'node_id': node['node_id'],
                'product_id': prod['product_ID'],
                'stock_on_hand': max(0, stock - sales),
                'units_sold': sales,
                # 'is_rto': False, # Returned order
                'units_rto': units_rto, 
                'rto_reason': rto_reason,
                'processing_lag_hours': np.random.uniform(0.5, 4.0) # The "Phantom Inventory" seed
            })

df_inventory = pd.DataFrame(records)
df_inventory.to_csv('02_Test_Inventory_baseline.csv', index = False)
print("Success: Baseline 31 days inventory state generated")

Success: Baseline 31 days inventory state generated
